In [ ]:
!pip install langchain faiss-cpu sentence-transformers huggingface-hub transformers langchain-community langchain-huggingface

In [ ]:
# TO AVOID WARNINGS
from langchain_core._api.deprecation import LangChainDeprecationWarning
import warnings

warnings.filterwarnings("ignore", category=LangChainDeprecationWarning)

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

/tmp/ipykernel_4367/2970152537.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
print(os.listdir('/content/drive/MyDrive'))

['PGT', 'Classroom', 'Cathrin Benz E J', 'Untitled presentation.pptx', 'அகப்பொருள்.pptx', 'Document from E.J Cathrin Benz (8)', 'Document from E.J Cathrin Benz (7)', 'Document from E.J Cathrin Benz (6)', 'Document from E.J Cathrin Benz (5)', 'Document from E.J Cathrin Benz (4)', 'Document from E.J Cathrin Benz (3)', 'Document from E.J Cathrin Benz (2)', 'Document from E.J Cathrin Benz (1)', 'Document from E.J Cathrin Benz', 'IMG-20230908-WA0033.jpg', 'Cathrin Benz E J (1).png', 'Mahizha mam 1', 'Downloads', 'python_file', 'Desktop', 'Important_Attachments.zip', 'Colab Notebooks', 'how to solve give video (1).gdoc', 'how to solve give video.gdoc', 'Screenshot_2025-07-04-18-15-25-40_c31b32364ce19ca8fcd150a417ecce58.jpg', 'screenshot.jpg', 'RESUME cathrin#.pdf', 'CATHRIN RESUME @#.pdf', 'DIABETES']


In [ ]:
from langchain_community.document_loaders import DirectoryLoader

# loading the notepad files from the directory
loader = DirectoryLoader('/content/drive/MyDrive/DIABETES',
glob="*.txt", loader_cls=TextLoader)
docs = loader.load()

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

chunks = text_splitter.split_documents(docs)

In [ ]:
from langchain_community.vectorstores import FAISS # database

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
embeddings

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [ ]:
db = FAISS.from_documents(chunks, embeddings)
docs = db.docstore._dict

for i, (doc_id, doc) in enumerate(docs.items()):
    print(f"Document {i+1}")
    print("ID:", doc_id)
    print("Content:")
    print(doc.page_content)
    print("-" * 60)

Document 1
ID: b8043f5f-5afb-4e29-8f9b-01afe8aa23bd
Content:
Diabetes
Diabetes is a common condition that affects people of all ages. There are several forms of diabetes. Type 2 is the most common. A combination of treatment strategies can help you manage the condition to live a healthy life and prevent complications.
What is diabetes?
Diabetes is a condition that happens when your blood sugar (glucose) is too high. It develops when your pancreas doesn’t make enough insulin or any at all, or when your body isn’t responding to the effects of insulin properly. Diabetes affects people of all ages. Most forms of diabetes are chronic (lifelong), and all forms are manageable with medications and/or lifestyle changes.
------------------------------------------------------------
Document 2
ID: 97289617-6e47-4a49-92e3-517851bb04b0
Content:
Glucose (sugar) mainly comes from carbohydrates in your food and drinks. It’s your body’s go-to source of energy. Your blood carries glucose to all your body

In [ ]:
vector = embeddings.embed_query("What is DIABETES?")
vector

[-0.02640272118151188,
 0.1092282086610794,
 -0.0924384668469429,
 0.10056494176387787,
 -0.052684225142002106,
 0.0275755375623703,
 0.15084287524223328,
 0.04183783382177353,
 0.005909809842705727,
 0.016620280221104622,
 -0.06484372913837433,
 0.036097243428230286,
 -0.06803157180547714,
 -0.035724055022001266,
 -0.11286160349845886,
 -0.06549479067325592,
 -0.03648454323410988,
 -0.03246447071433067,
 0.02253791317343712,
 0.0363176055252552,
 0.07742998003959656,
 0.0942320004105568,
 0.0426589697599411,
 0.01791679672896862,
 0.00946247112005949,
 -0.0127179604023695,
 0.047288425266742706,
 -0.01419329084455967,
 -0.06501471996307373,
 0.006886061280965805,
 -0.029131140559911728,
 0.06316543370485306,
 0.039168406277894974,
 0.048380784690380096,
 -0.08704426884651184,
 0.10905706137418747,
 0.02928803861141205,
 -0.016923055052757263,
 -0.09041676670312881,
 -0.06801260262727737,
 0.03519459068775177,
 -0.03018549457192421,
 -0.00556561816483736,
 0.07018369436264038,
 0.06989

In [ ]:
from transformers import pipeline
import torch

pipe = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-3B-Instruct", # Changed MODEL
    device=0 if torch.cuda.is_available() else -1,
    max_new_tokens=200,
    temperature=0.0,
    do_sample=False,
    return_full_text=False
)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.faiss import DistanceStrategy

db = FAISS.from_documents(
    chunks,
    embedding=embeddings,
    distance_strategy=DistanceStrategy.COSINE,
    normalize_L2=True,
)

/usr/local/lib/python3.12/dist-packages/langchain_community/vectorstores/faiss.py:233: UserWarning: Normalizing L2 is not applicable for metric type: DistanceStrategy.COSINE
  warnings.warn(


In [ ]:
retriever = db.as_retriever(
    search_type="similarity",
    search_kwargs={
        "k": 1
    }
)

In [ ]:
from langchain_huggingface import HuggingFacePipeline
llm = HuggingFacePipeline(pipeline=pipe)

In [ ]:
def self_rag_part(question: str):
    # Retrieve context
    context = "\n\n".join(
        doc.page_content for doc in retriever.invoke(question)  # IT PASSES THE QUESTION
    )

    # Generate answer
    answer_prompt = f"""
Answer the question using only the provided context.
If the answer is not in the context, say "I don't have enough information from the provided context."
Keep the answer within 2-3 lines.

Context:
{context}

Question:
{question}

Answer:
"""

    answer = llm.invoke(answer_prompt).strip()

    # Verify answer
    critique_prompt = f"""
Context:
{context}

Question:
{question}

Answer:
{answer}

Is the answer fully supported by the context?
Reply with ONLY YES or NO.
"""

    critique = llm.invoke(critique_prompt).strip().upper()

    # Revise if needed
    if critique == "NO":
        revise_prompt = f"""
Using only the context, rewrite the answer correctly.
Keep it within 2-3 lines.

Context:
{context}

Question:
{question}

Answer:
"""
        return llm.invoke(revise_prompt).strip()

    return answer

In [ ]:
print("Medical RAG Chatbot")
print("Type 'exit' to quit.\n")

while True:
    question = input(" Enter your medical question: ")

    # Exit condition
    if question.lower() in ["exit", "quit", "bye"]:
        print("Thank you! Stay healthy.")
        break

    # Get answer from your RAG
    answer = self_rag_part(question)

    # Display answer

    print("\n Answer:")
    print(answer)
    print("-" * 60)